# PDF RAG — Pinecone + Ollama + RAGAS (2 LLMs)

This evaluation notebook is designed to use the **same Pinecone vector space as the PDF RAG application**:

- Chunking already performed by the application: **300 tokens** (and the project's configured overlap; previously documented as 50).
- Pinecone index: `documents-rag` (confirm against your actual index name).
- Pinecone integrated embedding model: **`llama-text-embed-v2`**.
- Embedding dimension: **1024**.
- Pinecone namespace: `user_{USER_ID}`.
- Stored text metadata field: `chunk_text`.
- Retrieval: top 5 chunks.
- LLM 1 (answer generation): `llama3.2:3b`.
- LLM 2 (RAGAS judge): `qwen2.5:3b`.

**Important:** This notebook does not re-chunk or re-embed your PDF. It searches the vectors already stored by your application. Pinecone's integrated embedding configuration applies `llama-text-embed-v2` to the query when we search the `chunk_text` field. Re-embedding the document with another model would create a different vector space and invalidate a fair comparison.

The notebook checks index dimension, namespace, filter, retrieved chunk text, generated answers, and the judge output before interpreting Faithfulness.

## 1. Install dependencies

Run this cell once in the same Python environment where you run the notebook.

In [1]:
%pip install -U pinecone ragas datasets pandas python-dotenv requests langchain-core langchain-ollama

   ---------------------------------------- 0.0/591.1 kB ? eta -:--:--
   ---------------------------------------- 591.1/591.1 kB 7.5 MB/s  0:00:00

  Attempting uninstall: langchain-protocol

    Found existing installation: langchain-protocol 0.0.21

    Uninstalling langchain-protocol-0.0.21:

      Successfully uninstalled langchain-protocol-0.0.21

  Attempting uninstall: langchain-core

    Found existing installation: langchain-core 1.8.0

   ------------- -------------------------- 1/3 [langchain-core]
    Uninstalling langchain-core-1.8.0:
   ------------- -------------------------- 1/3 [langchain-core]
      Successfully uninstalled langchain-core-1.8.0
   ------------- -------------------------- 1/3 [langchain-core]
   ------------- -------------------------- 1/3 [langchain-core]
   ------------- -------------------------- 1/3 [langchain-core]
   ------------- -------------------------- 1/3 [langchain-core]
   ------------- -------------------------- 1/3 [langchain-core]
   


[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Configure your existing project

Set your Pinecone API key in `.env` as `PINECONE_KEY=...`. Do not paste secrets into the notebook.

Check the index name and user/document IDs below. The defaults reflect the project notebook and the chunk metadata example you shared (`document_id = 1`, `file_name = Python Tutorial.pdf`).

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_KEY")
INDEX_NAME = os.getenv("PINECONE_INDEX_NAME", "documents-rag")
USER_ID = os.getenv("RAG_EVAL_USER_ID", "3")
NAMESPACE = f"user_{USER_ID}"

# The example chunk you pasted has document_id=1.
DOCUMENT_ID = os.getenv("RAG_EVAL_DOCUMENT_ID", "1")
# Your pasted metadata displays document_id as 1 (numeric). Change to "string"
# only if Pinecone metadata stores it as the text "1".
DOCUMENT_ID_TYPE = os.getenv("RAG_EVAL_DOCUMENT_ID_TYPE", "number")

TOP_K = 5
EXPECTED_DIMENSION = 1024
PINECONE_EMBED_MODEL = "llama-text-embed-v2"
CHUNK_SIZE_TOKENS = 300

GENERATION_MODEL = os.getenv("RAG_GENERATION_MODEL", "llama3.2:3b")
JUDGE_MODEL = os.getenv("RAG_JUDGE_MODEL", "qwen2.5:3b")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")

assert PINECONE_API_KEY, "Missing PINECONE_KEY in .env"
assert GENERATION_MODEL != JUDGE_MODEL, "Choose two different Ollama LLM models."

print("Index:", INDEX_NAME)
print("Namespace:", NAMESPACE)
print("Document ID:", DOCUMENT_ID, "| metadata type:", DOCUMENT_ID_TYPE)
print("Pinecone integrated embedding model:", PINECONE_EMBED_MODEL)
print("Expected embedding dimension:", EXPECTED_DIMENSION)
print("Existing chunk size (tokens):", CHUNK_SIZE_TOKENS)
print("Answer LLM:", GENERATION_MODEL)
print("RAGAS judge LLM:", JUDGE_MODEL)

Index: documents-rag
Namespace: user_3
Document ID: 1 | metadata type: number
Pinecone integrated embedding model: llama-text-embed-v2
Expected embedding dimension: 1024
Existing chunk size (tokens): 300
Answer LLM: llama3.2:3b
RAGAS judge LLM: qwen2.5:3b


## 3. Connect to Pinecone and validate the vector index

This checks the index's vector dimension. The integrated embedding model is configured on the Pinecone index's embedding field; this notebook sends text queries to that field rather than creating local query vectors.

If the dimension is not 1024, stop and check the Pinecone index configuration before evaluating.

In [3]:
from pinecone import Pinecone

pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(INDEX_NAME)

stats = index.describe_index_stats()
print(stats)

dimension = getattr(stats, "dimension", None)
if dimension is None and isinstance(stats, dict):
    dimension = stats.get("dimension")

if dimension is not None:
    print("Pinecone dimension:", dimension)
    if int(dimension) != EXPECTED_DIMENSION:
        raise ValueError(
            f"Expected {EXPECTED_DIMENSION} dimensions for {PINECONE_EMBED_MODEL}, "
            f"but this index reports {dimension}. Check the index before continuing."
        )
else:
    print("Could not read dimension from this SDK response. Verify in Pinecone console: 1024.")

DescribeIndexStatsResponse(dimension=1024, total_vector_count=351, metric='cosine', namespaces=1)
Pinecone dimension: 1024


## 4. Questions and human-checked reference answers

Keep the references grounded in the same PDF. The reference answer is not sent to the generation model; it is used by reference-based evaluation metrics.

In [4]:
evaluation_questions = [
    {
        "question": "What are class objects in Python?",
        "reference": (
            "Class objects support attribute references and instantiation. "
            "Attribute references use the standard obj.name syntax and can access "
            "names in the class namespace. Class instantiation creates a new instance."
        )
    },
    {
        "question": "What are the main features of the if statement?",
        "reference": (
            "An if statement can have zero or more elif parts and an optional else part. "
            "elif means else if and helps avoid excessive indentation. "
            "An if/elif sequence can replace switch or case statements in other languages."
        )
    },
    {
        "question": "How does Python handle errors and exceptions?",
        "reference": (
            "When an error occurs, Python prints an error message and stack trace. "
            "An exception handled by an except clause is not treated as an unhandled "
            "error in that context. KeyboardInterrupt can be raised when the user "
            "interrupts a running command and may be handled by a try statement."
        )
    }
]
print("Questions:", len(evaluation_questions))

Questions: 3


## 5. Pinecone retrieval using the SAME integrated embedding field

`TextQuery(query=..., fields=["chunk_text"])` tells Pinecone to embed the query using the integrated embedding model configured for the `chunk_text` field. Do not use a different local embedding model for retrieval.

The code below handles the Pinecone document-search response and prints IDs, scores and metadata. It first provides a switch to disable the document filter for diagnosis.

In [5]:
from pinecone import TextQuery

USE_DOCUMENT_FILTER = True

def build_document_filter():
    if not USE_DOCUMENT_FILTER:
        return None
    if DOCUMENT_ID_TYPE == "number":
        return {"document_id": {"$eq": int(DOCUMENT_ID)}}
    if DOCUMENT_ID_TYPE == "string":
        return {"document_id": {"$eq": str(DOCUMENT_ID)}}
    raise ValueError("DOCUMENT_ID_TYPE must be 'number' or 'string'.")

def retrieve_matches(question: str, top_k: int = TOP_K):
    kwargs = {
        "namespace": NAMESPACE,
        "score_by": [TextQuery(query=question, fields=["chunk_text"])],
        "top_k": top_k,
        "include_fields": [
            "chunk_text", "title", "file_name", "document_id",
            "user_id", "chunk_id", "chunk_index"
        ]
    }
    metadata_filter = build_document_filter()
    if metadata_filter is not None:
        kwargs["filter"] = metadata_filter

    result = index.documents.search(**kwargs)
    matches = getattr(result, "matches", None)
    if matches is None and isinstance(result, dict):
        matches = result.get("matches", [])
    return list(matches or [])

def match_value(match, key, default=None):
    if isinstance(match, dict):
        return match.get(key, default)
    value = getattr(match, key, default)
    return value

def match_metadata(match):
    metadata = match_value(match, "metadata", {}) or {}
    if hasattr(metadata, "to_dict"):
        metadata = metadata.to_dict()
    return dict(metadata) if isinstance(metadata, dict) else {}

def match_text(match):
    # Pinecone document search may expose fields at top-level or in metadata.
    text = match_value(match, "chunk_text")
    if not text:
        text = match_metadata(match).get("chunk_text")
    return str(text).strip() if text else ""

def retrieve_contexts(question: str, top_k: int = TOP_K):
    matches = retrieve_matches(question, top_k)
    contexts = [match_text(m) for m in matches]
    return [c for c in contexts if c]

## 6. Retrieval diagnostic — inspect before generating answers

For the first question, inspect the returned chunks. If the query is about `if` statements but results are about classes, imports or garbage collection, do not run RAGAS yet.

If the filtered search looks wrong, set `USE_DOCUMENT_FILTER = False` and rerun this cell to test whether the document filter is the cause. Also verify the namespace and document ID in Pinecone.

In [6]:
for item in evaluation_questions:
    q = item["question"]
    matches = retrieve_matches(q)
    print("\\n" + "=" * 100)
    print("QUESTION:", q)
    print("MATCH COUNT:", len(matches))
    for i, m in enumerate(matches, 1):
        print(f"\\n--- MATCH {i} ---")
        print("ID:", match_value(m, "id"))
        print("SCORE:", match_value(m, "score"))
        print("METADATA:", match_metadata(m))
        print("TEXT:", match_text(m)[:1400])

\n====================================================================================================
QUESTION: What are class objects in Python?
MATCH COUNT: 5
\n--- MATCH 1 ---
ID: doc_1_chunk_169
SCORE: 9.008644
METADATA: {}
TEXT: passed on to __init__(). For example,
>>> class Complex:
... def __init__(self, realpart, imagpart):
... self.r = realpart
... self.i = imagpart
...
>>> x = Complex(3.0, -4.5)
>>> x.r, x.i
(3.0, -4.5)
9.3.3 Instance Objects
Now what can we do with instance objects? The only operations understood by instance objects are attribute
references. There are two kinds of valid attribute names, data attributes and methods.
data attributes correspond to “instance variables” in Smalltalk, and to “data members” in C++. Data
attributes need not be declared; like local variables, they spring into existence when they are ﬁrst assigned
to. For example, if x is the instance of MyClass created above, the following piece of code will print the
value 16, without leaving a tr

## 7. Configure the two Ollama LLMs

- **LLM 1 — answer generator:** `llama3.2:3b`
- **LLM 2 — RAGAS judge:** `qwen2.5:3b`

Install both locally before running:
```bash
ollama pull llama3.2:3b
ollama pull qwen2.5:3b
```

The judge is separate from the generator, so it can independently inspect the answer against the retrieved context. A 3B judge is still relatively small; if it repeatedly returns malformed judgments or suspicious all-zero scores, compare with a stronger judge model.

In [7]:
import requests

def check_ollama_model(model_name: str):
    response = requests.post(
        f"{OLLAMA_BASE_URL}/api/show",
        json={"name": model_name},
        timeout=60
    )
    response.raise_for_status()
    return True

for model_name in [GENERATION_MODEL, JUDGE_MODEL]:
    check_ollama_model(model_name)
    print("Ollama model available:", model_name)

Ollama model available: llama3.2:3b
Ollama model available: qwen2.5:3b


## 8. Generate grounded answers with LLM 1

The generator is instructed to use only retrieved context and to say when the context is insufficient. It does not see the reference answer.

In [8]:
def generate_answer(question: str, contexts: list[str]) -> str:
    if not contexts:
        return "The retrieved document context does not contain enough information to answer this question."

    context_text = "\\n\\n---\\n\\n".join(contexts)
    prompt = '''You are a PDF question-answering assistant.
                    Answer the question using ONLY the supplied context.
                    Do not use outside knowledge or invent facts.
                    If the context does not contain the answer, say so clearly.
                    Keep the answer concise and directly answer the question.

                    CONTEXT:
                    {context_text}

                    QUESTION:
                    {question}

                    ANSWER:
                    '''

    response = requests.post(
        f"{OLLAMA_BASE_URL}/api/generate",
        json={
            "model": GENERATION_MODEL,
            "prompt": prompt,
            "stream": False,
            "options": {"temperature": 0}
        },
        timeout=240
    )
    response.raise_for_status()
    return response.json()["response"].strip()

## 9. Build the evaluation rows

The retrieved contexts here come directly from Pinecone's existing vector index. No chunks are recreated in this notebook.

In [9]:
rows = []
for item in evaluation_questions:
    question = item["question"]
    contexts = retrieve_contexts(question, TOP_K)
    answer = generate_answer(question, contexts)

    rows.append({
        "user_input": question,
        "retrieved_contexts": contexts,
        "response": answer,
        "reference": item["reference"]
    })

print("Evaluation rows:", len(rows))

Evaluation rows: 3


## 10. Quality gate — stop if retrieval is wrong

Review each row before evaluating. A faithfulness score of zero can be correct if the answer contains claims unsupported by the retrieved chunks. It can also be an LLM-judge failure. First establish whether the contexts are actually relevant.

In [10]:
import pandas as pd

for row in rows:
    print("\\n" + "=" * 100)
    print("QUESTION:", row["user_input"])
    print("ANSWER:", row["response"])
    print("CONTEXT COUNT:", len(row["retrieved_contexts"]))
    for j, ctx in enumerate(row["retrieved_contexts"], 1):
        print(f"\\nCONTEXT {j}:\\n{ctx[:1800]}")

if any(len(row["retrieved_contexts"]) == 0 for row in rows):
    raise RuntimeError("At least one question has no retrieved chunks. Fix retrieval before RAGAS.")

preview = pd.DataFrame([
    {
        "question": r["user_input"],
        "num_contexts": len(r["retrieved_contexts"]),
        "answer": r["response"],
        "reference": r["reference"]
    } for r in rows
])
display(preview)

\n====================================================================================================
QUESTION: What are class objects in Python?
ANSWER: I'm ready to assist. What is the context and question?
CONTEXT COUNT: 5
\nCONTEXT 1:\npassed on to __init__(). For example,
>>> class Complex:
... def __init__(self, realpart, imagpart):
... self.r = realpart
... self.i = imagpart
...
>>> x = Complex(3.0, -4.5)
>>> x.r, x.i
(3.0, -4.5)
9.3.3 Instance Objects
Now what can we do with instance objects? The only operations understood by instance objects are attribute
references. There are two kinds of valid attribute names, data attributes and methods.
data attributes correspond to “instance variables” in Smalltalk, and to “data members” in C++. Data
attributes need not be declared; like local variables, they spring into existence when they are ﬁrst assigned
to. For example, if x is the instance of MyClass created above, the following piece of code will print the
value 16, without leavin

,question,num_contexts,answer,reference
0,What are class objects in Python?,5,I'm ready to assist. What is the context and q...,Class objects support attribute references and...
1,What are the main features of the if statement?,5,I'm ready to assist. What is the context and q...,An if statement can have zero or more elif par...
2,How does Python handle errors and exceptions?,5,I'm ready to assist. What is the context and q...,"When an error occurs, Python prints an error m..."


## 11. Configure RAGAS with LLM 2 as the judge

RAGAS Faithfulness uses the judge LLM to break the response into claims and check those claims against the retrieved context. We use the separate Qwen model as that judge.

RAGAS Answer Relevancy also needs embeddings. This notebook uses `nomic-embed-text` locally for that metric only. It does **not** affect Pinecone retrieval and does not replace the `llama-text-embed-v2` index embeddings.

In [11]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

judge_chat_model = ChatOllama(
    model=JUDGE_MODEL,
    base_url=OLLAMA_BASE_URL,
    temperature=0
)
judge_llm = LangchainLLMWrapper(judge_chat_model)

# Embedding model is used only inside RAGAS for embedding-based evaluation metrics.
# It is not used to query or modify Pinecone.
ragas_embedding_model = LangchainEmbeddingsWrapper(
    OllamaEmbeddings(
        model=os.getenv("RAGAS_EMBEDDING_MODEL", "nomic-embed-text"),
        base_url=OLLAMA_BASE_URL
    )
)

c:\Users\KeerthanaV\Desktop\Projects\rag_app\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\KeerthanaV\AppData\Local\Temp\ipykernel_25972\3586109588.py:10: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  judge_llm = LangchainLLMWrapper(judge_chat_model)
C:\Users\KeerthanaV\AppData\Local\Temp\ipykernel_25972\3586109588.py:14: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, Google

## 12. Run RAGAS Faithfulness and supporting metrics

This notebook uses the current RAGAS metric interface. If your installed RAGAS version has breaking API changes, check its version and update the metric imports accordingly rather than silently changing the evaluation setup.

In [12]:
import ragas
print("RAGAS version:", getattr(ragas, "__version__", "version not exposed"))

from ragas import EvaluationDataset, evaluate
from ragas.metrics import (
    Faithfulness,
    ResponseRelevancy,
    LLMContextPrecisionWithReference,
    LLMContextRecall
)

dataset = EvaluationDataset.from_list(rows)

metrics = [
    Faithfulness(),
    ResponseRelevancy(),
    LLMContextPrecisionWithReference(),
    LLMContextRecall()
]

from ragas.run_config import RunConfig
run_config = RunConfig(max_workers=1, max_retries=2, timeout=240)

results = evaluate(
    dataset=dataset,
    metrics=metrics,
    llm=judge_llm,
    embeddings=ragas_embedding_model,
    run_config=run_config
)

scores = results.to_pandas()
display(scores)

C:\Users\KeerthanaV\AppData\Local\Temp\ipykernel_25972\1760516696.py:5: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
C:\Users\KeerthanaV\AppData\Local\Temp\ipykernel_25972\1760516696.py:5: DeprecationWarning: Importing ResponseRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import (
C:\Users\KeerthanaV\AppData\Local\Temp\ipykernel_25972\1760516696.py:5: DeprecationWarning: Importing LLMContextPrecisionWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecisionWithReference
  from ragas.metrics import (
C:\U

RAGAS version: 0.4.4


Evaluating: 100%|██████████| 12/12 [10:47<00:00, 54.00s/it]


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,llm_context_precision_with_reference,context_recall
0,What are class objects in Python?,"[passed on to __init__(). For example,\n>>> cl...",I'm ready to assist. What is the context and q...,Class objects support attribute references and...,1.0,0.409466,0.804167,1.000000
1,What are the main features of the if statement?,[.effects\nimport echo.\nYou can also write re...,I'm ready to assist. What is the context and q...,An if statement can have zero or more elif par...,0.0,0.418772,0.333333,1.000000
2,How does Python handle errors and exceptions?,[now error messages haven’t been more than men...,I'm ready to assist. What is the context and q...,"When an error occurs, Python prints an error m...",1.0,0.405608,1.000000,0.666667


## 13. Inspect Faithfulness row by row

Do not look only at the average. For every zero-score answer, compare the answer's factual claims with the exact retrieved contexts. If the answer includes facts not in context, the zero may be valid.

In [13]:
if "faithfulness" not in scores.columns:
    raise KeyError(f"Faithfulness column missing. Actual columns: {list(scores.columns)}")

display(scores[["user_input", "response", "retrieved_contexts", "faithfulness"]])

print("Average faithfulness:", scores["faithfulness"].mean())

,user_input,response,retrieved_contexts,faithfulness
0,What are class objects in Python?,I'm ready to assist. What is the context and q...,"[passed on to __init__(). For example,\n>>> cl...",1.0
1,What are the main features of the if statement?,I'm ready to assist. What is the context and q...,[.effects\nimport echo.\nYou can also write re...,0.0
2,How does Python handle errors and exceptions?,I'm ready to assist. What is the context and q...,[now error messages haven’t been more than men...,1.0


Average faithfulness: 0.6666666666666666


## 14. Troubleshooting an all-zero Faithfulness score

Use this decision sequence:

1. **Wrong chunks:** Fix namespace, document filter, `chunk_text` field, or retrieval query. Do not change the embedding model if the index was built with `llama-text-embed-v2`.
2. **Right chunks, unsupported answer:** Improve the answer prompt or generator; keep temperature at zero.
3. **Right chunks and clearly supported answer, but still zero:** Test the judge separately. A small judge may fail claim extraction or structured output. Temporarily use a stronger judge (for example `qwen2.5:7b`) while keeping the generator unchanged.
4. **One question only:** Inspect that specific response and contexts; average scores can hide per-question issues.

**Do not use `llama-text-embed-v2` as an Ollama model name.** It is the Pinecone integrated embedding model configured on the index. The query text is embedded by Pinecone when searching the configured `chunk_text` field.

## 15. Save scores

In [14]:
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = f"ragas_pdf_scores_{timestamp}.csv"
scores.to_csv(output_file, index=False)
print("Saved:", output_file)

Saved: ragas_pdf_scores_20261124_151132.csv
